# Simulation 1: matrix completion via PathFinder joint decomposition

Two modalities across three domains arranged on a 2x3 grid. Four cells are
observed, two are held out. PathFinder jointly factorises the observed
cells and predicts the missing ones. We sweep over noise level and number
of components to characterise recovery quality.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from pathfinder.decomp import JointOuterDecomp

sns.set_theme(style='whitegrid', context='paper', font_scale=1.15,
              rc={'axes.edgecolor': '.15', 'grid.color': '.88'})

n_rows_A = [100, 200]
n_cols_S = [100, 200, 300]
true_rank = 20
noise_levels = [0.0001, 0.001, 0.01, 0.1]
n_components_list = [5, 10, 15, 20]

n_repeats = 5
n_restarts = 5
n_iter = 100

obs_pairs = [(0, 0), (0, 1), (1, 0), (1, 2)]
miss_pairs = [(0, 2), (1, 1)]

tab10 = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2',
         '#59a14f', '#edc949', '#af7aa1', '#ff9da7']
obs_colour, miss_colour = tab10[0], tab10[1]
FIGW, FIGH = 7, 5

## Core functions

`generate_ground_truth` builds a 2x3 grid of outer-product matrices from
Gaussian factors with additive Frobenius-normalised noise.
`run_single` fits PathFinder with multi-restart and returns $R^2$ for each
observed and missing cell.

In [ ]:
def generate_ground_truth(noise_level, rng):
    A = [rng.standard_normal((nr, true_rank)) for nr in n_rows_A]
    S = [rng.standard_normal((nc, true_rank)) for nc in n_cols_S]
    X_clean = {(i, j): A[i] @ S[j].T for i in range(2) for j in range(3)}
    X_noisy = {}
    for (i, j), C in X_clean.items():
        noise = rng.standard_normal(C.shape)
        noise = noise / np.linalg.norm(noise, 'fro') * np.linalg.norm(C, 'fro') * noise_level
        X_noisy[(i, j)] = C + noise
    return X_clean, X_noisy


def compute_r2(true, pred):
    return np.corrcoef(true.flatten(), pred.flatten())[0, 1] ** 2


def run_single(noise_level, n_comp, rng):
    X_clean, X_noisy = generate_ground_truth(noise_level, rng)
    Clist = [X_noisy[p] for p in obs_pairs]
    alpha = [p[0] for p in obs_pairs]
    beta = [p[1] for p in obs_pairs]

    best_loss, best = np.inf, None
    for r in range(n_restarts):
        np.random.seed(int(rng.integers(1e9)) + r * 7)
        d = JointOuterDecomp(n_components=n_comp, n_iter=n_iter, alpha=1e-5,
                             batch_size=10, init_type='random',
                             update_fraction=1.0, verbose=False)
        d.fit(Clist, alpha, beta)
        loss = np.mean(d._loss[-1])
        if loss < best_loss:
            best_loss, best = loss, d

    preds = best.predict()
    r2s = {p: compute_r2(X_clean[p], preds[idx]) for idx, p in enumerate(obs_pairs)}
    for (i, j) in miss_pairs:
        r2s[(i, j)] = compute_r2(X_clean[(i, j)], best._A[i] @ best._S[j].T)
    return r2s, best, X_clean

## Panel A: grid configuration

Observed cells in blue, missing cells in orange.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(FIGW, FIGH))
fig.subplots_adjust(hspace=0.25, wspace=0.2)
for i in range(2):
    for j in range(3):
        ax = axes[i, j]
        is_obs = (i, j) in obs_pairs
        fc = obs_colour if is_obs else miss_colour
        alpha_val = 0.85 if is_obs else 0.50
        rect = patches.FancyBboxPatch(
            (0.05, 0.05), 0.9, 0.9, boxstyle='round,pad=0.04',
            transform=ax.transAxes, facecolor=fc, edgecolor='white',
            linewidth=2.5, alpha=alpha_val)
        ax.add_patch(rect)
        status = 'Observed' if is_obs else 'Missing'
        ax.text(0.5, 0.55, f'{n_rows_A[i]}\u00d7{n_cols_S[j]}', transform=ax.transAxes,
                ha='center', va='center', fontsize=16, color='white', fontweight='bold')
        ax.text(0.5, 0.33, f'({status})', transform=ax.transAxes,
                ha='center', va='center', fontsize=10, color='white', alpha=0.9)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
        if i == 0:
            ax.set_title(f'Domain {j}', fontsize=11, fontweight='bold')
        if j == 0:
            ax.set_ylabel(f'Modality {i}', fontsize=11, fontweight='bold', labelpad=10)
plt.show()

## Run the noise x rank sweep

`n_repeats` repetitions per (noise, rank) cell. This is the slow cell.

In [ ]:
results = {}
for ni, noise in enumerate(noise_levels):
    for ci, n_comp in enumerate(n_components_list):
        reps = []
        for rep in range(n_repeats):
            rng = np.random.default_rng(seed=rep * 1000 + ni * 100 + ci)
            r2s, _, _ = run_single(noise, n_comp, rng)
            reps.append(r2s)
        results[(ni, ci)] = reps
        obs_med = np.median([np.mean([r[p] for p in obs_pairs]) for r in reps])
        miss_med = np.median([np.mean([r[p] for p in miss_pairs]) for r in reps])
        print(f'noise={noise:.4f}  r={n_comp:2d} | obs R\u00b2={obs_med:.4f}  miss R\u00b2={miss_med:.4f}')

## Panel B: true vs reconstructed scatter plots

Single example run at `noise=0.01`, `n_comp=20`.

In [ ]:
rng_ex = np.random.default_rng(seed=12345)
_, best_ex, X_clean_ex = run_single(0.01, 20, rng_ex)
preds_ex = {p: best_ex.predict()[idx] for idx, p in enumerate(obs_pairs)}
for (i, j) in miss_pairs:
    preds_ex[(i, j)] = best_ex._A[i] @ best_ex._S[j].T

fig, axes = plt.subplots(2, 3, figsize=(FIGW, FIGH))
fig.subplots_adjust(hspace=0.15, wspace=0.15)
plot_rng = np.random.default_rng(0)
for i in range(2):
    for j in range(3):
        ax = axes[i, j]
        is_obs = (i, j) in obs_pairs
        colour = obs_colour if is_obs else miss_colour
        tf = X_clean_ex[(i, j)].flatten()
        pf = preds_ex[(i, j)].flatten()
        if len(tf) > 5000:
            idx = plot_rng.choice(len(tf), 5000, replace=False)
            tf, pf = tf[idx], pf[idx]
        ax.scatter(tf, pf, s=2, alpha=0.3, color=colour,
                   edgecolors='none', rasterized=True)
        lim = max(np.abs(tf).max(), np.abs(pf).max()) * 1.05
        ax.plot([-lim, lim], [-lim, lim], '-', color='#999999',
                linewidth=0.8, zorder=0)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
        if i == 1:
            ax.set_xlabel('True', fontsize=14)
        else:
            ax.set_xticklabels([])
        if j == 0:
            ax.set_ylabel('Reconstructed', fontsize=14)
        else:
            ax.set_yticklabels([])
        ax.tick_params(labelsize=7)
plt.show()

## Panel C: R² vs number of components (single noise level)

Recovery saturates once the number of fitted components meets the true rank.

In [ ]:
def summarise(reps, pairs):
    vals = [np.mean([r[p] for p in pairs]) for r in reps]
    return (np.median(vals),
            np.percentile(vals, 25),
            np.percentile(vals, 75))


ni = 2  # noise = 0.01
col1, col2 = tab10[0], tab10[1]
obs_stats = [summarise(results[(ni, ci)], obs_pairs) for ci in range(len(n_components_list))]
miss_stats = [summarise(results[(ni, ci)], miss_pairs) for ci in range(len(n_components_list))]
obs_med, obs_lo, obs_hi = map(list, zip(*obs_stats))
miss_med, miss_lo, miss_hi = map(list, zip(*miss_stats))

fig, ax = plt.subplots(figsize=(FIGW, FIGH))
ax.plot(n_components_list, obs_med, 'o-', color=col1, markersize=6, linewidth=1.8)
ax.fill_between(n_components_list, obs_lo, obs_hi, alpha=0.12, color=col1)
ax.plot(n_components_list, miss_med, 's--', color=col2, markersize=6, linewidth=1.8)
ax.fill_between(n_components_list, miss_lo, miss_hi, alpha=0.12, color=col2)
ax.axvline(x=true_rank, color='grey', linestyle=':', linewidth=0.8, alpha=0.6)
ax.set_xlabel('Number of components', fontsize=14)
ax.set_ylabel('R\u00b2', fontsize=14)
ax.set_ylim(-0.05, 1.05)
handles = [plt.Line2D([0], [0], color=col1, marker='o', linestyle='-'),
           plt.Line2D([0], [0], color=col2, marker='s', linestyle='--')]
ax.legend(handles, ['Observed', 'Missing'], loc='center right', fontsize=8, frameon=False)
plt.show()

## Panel D: R² vs noise level (one colour per rank)

In [ ]:
fig, ax = plt.subplots(figsize=(FIGW, FIGH))
for ci, n_comp in enumerate(n_components_list):
    colour = tab10[ci]
    obs_stats = [summarise(results[(ni, ci)], obs_pairs) for ni in range(len(noise_levels))]
    miss_stats = [summarise(results[(ni, ci)], miss_pairs) for ni in range(len(noise_levels))]
    obs_med, obs_lo, obs_hi = map(list, zip(*obs_stats))
    miss_med, miss_lo, miss_hi = map(list, zip(*miss_stats))
    ax.plot(noise_levels, obs_med, 'o-', color=colour, markersize=6, linewidth=1.8)
    ax.fill_between(noise_levels, obs_lo, obs_hi, alpha=0.12, color=colour)
    ax.plot(noise_levels, miss_med, 's--', color=colour, markersize=6, linewidth=1.8)
    ax.fill_between(noise_levels, miss_lo, miss_hi, alpha=0.12, color=colour)

ax.set_xscale('log')
ax.set_xlabel('Noise level (\u03c3 / \u2016signal\u2016)')
ax.set_ylabel('R\u00b2')
ax.set_ylim(-0.05, 1.05)

h_comp = [plt.Line2D([0], [0], color=tab10[ci], marker='o') for ci in range(len(n_components_list))]
l_comp = [f'r = {n}' for n in n_components_list]
h_type = [plt.Line2D([0], [0], color='#555555', linestyle='-', marker='o'),
          plt.Line2D([0], [0], color='#555555', linestyle='--', marker='s')]
leg1 = ax.legend(h_comp, l_comp, loc='lower left', fontsize=8,
                 title='Components', title_fontsize=9,
                 frameon=True, fancybox=False, edgecolor='#cccccc')
ax.add_artist(leg1)
ax.legend(h_type, ['Observed', 'Missing'], loc='center left', fontsize=8,
          frameon=True, fancybox=False, edgecolor='#cccccc')
plt.show()